<a href="https://colab.research.google.com/github/DzeGra/reducto-financial-extraction/blob/main/reducto_financial_extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Reducto Financial Extraction — Amazon Q4 2024
**Use case:** an LP monitoring portfolio companies and public comparables needs earnings data in a standard Excel format. Today analysts key it in by hand.

**Pipeline:** PDF → Reducto upload → Reducto Extract (JSON schema, citations on) → validation → LP workbook

| Step | What happens |
|---|---|
| 1 | Setup and authentication |
| 2 | Upload the earnings release to Reducto |
| 3 | Load the extraction schema |
| 4 | Run the extraction (or reuse a completed job) |
| 5 | Inspect one field and its citation |
| 6 | Validate the numbers |
| 7 | Build the LP workbook |
| 8 | Download |

## 1. Setup and authentication
The API key lives in Colab Secrets (🔑 icon in the left sidebar) under the name `REDUCTO_API_KEY`, so it never appears in the notebook.

In [ ]:
!pip install -q reductoai openpyxl
!git clone -q https://github.com/DzeGra/reducto-financial-extraction.git 2>/dev/null || git -C reducto-financial-extraction pull -q

import sys, json
sys.path.append("reducto-financial-extraction/src")

from google.colab import userdata, files
from reducto import Reducto
from build_workbook import load_payload, values_from, kpi_records, validation_checks, build_workbook, source_page

client = Reducto(api_key=userdata.get("REDUCTO_API_KEY"))
print("Reducto client ready")

## 2. Upload the earnings release
Select `AMZN-Q4-2024-Earnings-Release.pdf` when prompted. Reducto returns a file reference that the extract call uses.

In [ ]:
uploaded = files.upload()
pdf_name = list(uploaded.keys())[0]

with open(pdf_name, "rb") as f:
    upload = client.upload(file=f)

print("Uploaded:", pdf_name)

## 3. Load the extraction schema
The schema is the contract with the LP: one field per number they asked for, each with a plain-English description of where it lives in the document and what unit to return.
It covers the income statement, segments, cash flow, balance sheet, Q1 2025 guidance, every KPI row in the supplemental tables, and qualitative partnerships.

In [ ]:
with open("reducto-financial-extraction/schema/amazon_q4_2024_schema.json") as f:
    instructions = json.load(f)

fields = instructions["schema"]["properties"]
print(len(fields), "fields in the schema")
print(json.dumps(fields["q4_2024_total_net_sales"], indent=2))

## 4. Run the extraction
`citations` returns the page, bounding box and confidence score for every value, which gives the LP an audit trail.
Set `REUSE_JOB_ID` to a completed job ID to skip re-running and save credits.

In [ ]:
REUSE_JOB_ID = "def02ff6-284a-4030-a18c-521cbd0a9e9b"

if REUSE_JOB_ID:
    response = client.job.get(REUSE_JOB_ID)
    job_id = REUSE_JOB_ID
else:
    response = client.extract.run(
        input=upload,
        instructions=instructions,
        settings={"citations": {"enabled": True}},
        parsing={"settings": {"persist_results": True}},
    )
    job_id = response.model_dump().get("job_id")

amazon_raw = load_payload(response)   # full result, citations included
amazon = values_from(amazon_raw)      # plain values for calculations

r = response.model_dump()
studio_link = r.get("studio_link") or (r.get("result") or {}).get("studio_link")

print("Job ID:", job_id)
print("Studio link:", studio_link)
print("Fields returned:", len(amazon_raw))
print("Not in source:", [k for k, v in amazon.items() if v is None])

## 5. Inspect one field and its citation
Every value carries the text it came from, the page, its position on the page and a confidence score. The same bounding boxes are what Reducto Studio highlights on the PDF: open the Studio link from step 4 and click `fy2024_aws_operating_income` to see it.

In [ ]:
item = amazon_raw["fy2024_aws_operating_income"]
cite = item["citations"][0]

print("Value:     ", item["value"])
print("Page:      ", cite["bbox"]["page"])
print("Position:  ", {k: round(cite["bbox"][k], 3) for k in ("left", "top", "width", "height")})
print("Confidence:", cite.get("granular_confidence"))
print("Source:    ", cite.get("content"))

## 6. Validate the numbers
Each check reconciles extracted numbers against other numbers in the same document:
segments must add up to totals, free cash flow must equal operating cash flow minus net capex,
and the supplemental tables (p.10) must agree with the financial statements (pp.5–6).

In [ ]:
kpis = kpi_records(amazon_raw)
checks = validation_checks(amazon, kpis)

for name, calculated, reported in checks:
    status = "PASS" if abs(calculated - reported) < 0.5 else "FAIL"
    print(f"{status}  {name}:  {calculated:,} vs {reported:,}")

## 7. Build the LP workbook
Tabs match the LP brief: **Financials, Segments, Cash Flow, Guidance, Notes**, plus KPIs, Disclosures, Validation and Extraction Audit.
Growth rates, margins and mix are Excel formulas, so an analyst can click any cell and see the math. Fields missing from the source are left out and listed in Notes.

In [ ]:
path, _ = build_workbook(
    amazon_raw,
    "Amazon_Q4_2024_LP_Analysis.xlsx",
    meta={"job_id": job_id, "n_fields": len(fields)},
)
print("Workbook ready:", path)

## 8. Download

In [ ]:
files.download(path)

## Results and technical decisions
- **Accuracy:** every extracted value was checked against the source PDF; all reconciliation checks pass.
- **Traceability:** citations are kept for every field (page, bounding box, confidence) and shown in the Extraction Audit tab.
- **Schema design:** field descriptions name the exact table and line item, the unit, and the sign convention, which removed ambiguity in earlier runs.
- **Separation of concerns:** extraction (Reducto), validation and workbook layout (`src/build_workbook.py`) are independent, so the same pipeline runs on the next company by swapping the schema.
- **Security:** the API key is read from Colab Secrets.